<a href="https://colab.research.google.com/github/Emanloloco/Sorry-ser/blob/main/Phase_4_Diabetes_Dataset_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This loads our Diabetes Dataset and Tuned Model

In [31]:
url = "https://raw.githubusercontent.com/Emanloloco/Sorry-ser/refs/heads/main/Diabetes%20Dataset%20(2)-1.csv"
df = pd.read_csv(url) #Load the Titanic dataset from a CSV file into a DataFrame https://www.kaggle.com/datasets/yasserh/titanic-dataset
df.head()

/tmp/ipykernel_10368/2211087005.py:2: DtypeWarning: Columns (35,36,37,38,39,40,41,42,43,44,60,61,62,63,64,65,66,67,68,69,70,71,72,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url) #Load the Titanic dataset from a CSV file into a DataFrame https://www.kaggle.com/datasets/yasserh/titanic-dataset


,id,age,Age Z-Score,gender,bmi,BMI Z-Score,glucose,Glucose Z-Score,blood_pressure,Blood Pressure Z-Score,...,Unnamed: 64,Unnamed: 65,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72,Unnamed: 73
0,1,69,0.744812,Male,23.30,-0.344105,170,0.956919,137,0.258796,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,32,-1.037692,Male,25.00,0.002805,184,1.328779,177,1.636011,...,Blood Pressure Z-Score,Cholesterol Z-Score,Heart Rate Z-Score,Sleep Hour Z-Score,Stress Level Z-Score,Steps per day Z-Score,Diet Score Z-Score,Work Hour Z-Score,Water Intake Z-Score,Insulin Z-Score
2,3,89,1.708327,Female,28.57,0.731317,87,-1.247682,164,1.188416,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,78,1.178394,Male,15.85,-1.864388,96,-1.008629,113,-0.567533,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,38,-0.748637,Female,35.74,2.194462,171,0.983480,122,-0.257660,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Removed joblib.load as we are training from scratch
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

# New imports for model training
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# Load dataset
df = pd.read_excel("/content/Diabetes Dataset (2)-1.xlsx")

y = df["diabetes"]
X = df.drop(columns=["diabetes"])
if "id" in X.columns:
    X = X.drop(columns=["id"])

# Same split as Phase 3
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define and Train Random Forest Pipeline
# Identify categorical and numerical columns
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include="object").columns.tolist()

# Create preprocessing pipelines for numerical and categorical features
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create the full pipeline with preprocessing and the Random Forest classifier
best_rf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('classifier', RandomForestClassifier(random_state=42))]) # Added random_state for reproducibility

# Train the model
print("Training Random Forest model...")
best_rf.fit(X_train, y_train)
print("Training complete.")

# Preds and probabilities
rf_pred = best_rf.predict(X_test)
rf_proba = best_rf.predict_proba(X_test)[:, 1]

FileNotFoundError: [Errno 2] No such file or directory: '/content/Diabetes Dataset (2)-1.xlsx'

In [ ]:
# ── DELIVERABLE 11: Class Distribution (Test Set) ───────────────────────────
counts = y_test.value_counts().sort_index()

color_map = {0: "steelblue", 1: "tomato"}
label_map = {0: "No Diabetes", 1: "Diabetes"}

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    [label_map[i] for i in counts.index],
    counts.values,
    color=[color_map[i] for i in counts.index],   # consistent class colors
    edgecolor="white",                             # consistent with histograms
)

# Annotate bars with counts
for bar, count in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        str(count),
        ha="center", va="bottom", fontsize=11,
    )

ax.set_title("Class Distribution — Test Set (Diabetes)")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

This bar chart helps us check whether the dataset is balanced or imbalanced.
This is important because class imbalance can affect model evaluation and may require metrics like Precision, Recall, and F1-score.

In [ ]:
# ── DELIVERABLE 10: Feature Histograms (split by target class) ───────────────
numeric_cols  = X.select_dtypes(include=["number"]).columns.tolist()
cols_to_plot  = [c for c in ["bmi", "glucose", "insulin",
                              "cholesterol", "age", "blood_pressure"]
                 if c in numeric_cols]

plot_df       = X_test.copy()
plot_df["diabetes"] = y_test.values

color_map  = {0: "steelblue", 1: "tomato"}
label_map  = {0: "No Diabetes", 1: "Diabetes"}

for col in cols_to_plot:
    fig, ax = plt.subplots(figsize=(7, 5))

    for cls in [0, 1]:
        subset = plot_df.loc[plot_df["diabetes"] == cls, col].dropna()
        ax.hist(
            subset,
            bins=30,                   # consistent with residual histogram
            alpha=0.5,                 # overlap visibility
            color=color_map[cls],
            edgecolor="white",         # consistent with residual histogram
            label=label_map[cls],
        )

    ax.set_title(f"Histogram of {col.replace('_', ' ').title()} by Diabetes Status")
    ax.set_xlabel(col.replace("_", " ").title())
    ax.set_ylabel("Frequency")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── DELIVERABLE 9: Box Plots per Feature (split by target class) ─────────────
cols_to_plot = X.select_dtypes(include=np.number).columns.tolist()  # explicit definition

plot_df = X_test.copy()
plot_df["diabetes"] = y_test.values

for col in cols_to_plot:
    nan_count = plot_df[col].isna().sum()
    if nan_count > 0:
        print(f"[INFO] {col}: {nan_count} NaN(s) dropped before plotting")

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(
        data=plot_df,
        x="diabetes",
        y=col,
        hue="diabetes",
        palette={0: "steelblue", 1: "tomato"},   # consistent class colors
        legend=False,
        ax=ax,
    )
    ax.set_title(f"Box Plot of {col.replace('_', ' ').title()} by Diabetes Status")
    ax.set_xlabel("Diabetes Status")
    ax.set_ylabel(col.replace("_", " ").title())
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No Diabetes", "Diabetes"])
    plt.tight_layout()
    plt.show()

In [ ]:
# ── DELIVERABLE 8: Pairwise Scatter Plots (colored by target class) ──────────
pairs = [("bmi", "glucose"), ("age", "blood_pressure"), ("steps_per_day", "bmi")]

label_map = {0: "No Diabetes", 1: "Diabetes"}
colors     = {0: "steelblue",  1: "tomato"}

for a, b in pairs:
    if a in X.columns and b in X.columns:
        fig, ax = plt.subplots(figsize=(7, 5))

        for cls in [0, 1]:
            mask = y_test.values == cls
            ax.scatter(
                X_test.loc[y_test.index[mask], a],
                X_test.loc[y_test.index[mask], b],
                s=10,
                alpha=0.5,
                color=colors[cls],
                label=label_map[cls],
            )

        ax.set_title(f"Scatter Plot: {a.replace('_', ' ').title()} vs {b.replace('_', ' ').title()}")
        ax.set_xlabel(a.replace("_", " ").title())
        ax.set_ylabel(b.replace("_", " ").title())
        ax.legend()
        plt.tight_layout()
        plt.show()

The BMI histogram looks close to a bell shape. Most values are concentrated around the mid-20s, and fewer people appear in very low or very high BMI ranges. This means BMI has a “common” middle range and fewer extreme values. There are small tails on both sides, which suggests some outliers but not too many.

The glucose histogram looks almost flat from about 70 to 200. This means glucose values are spread out fairly evenly across the whole range, and there is no strong peak.

The insulin histogram also looks almost uniform, meaning the values are spread across the range without one clear most common region. There is no strong distortion.

The cholesterol histogram appears fairly uniform across roughly 120 to 300. There are small ups and downs, but overall there is no strong peak. This means cholesterol values are fairly evenly represented across the range.

The age histogram looks mostly uniform across about 18 to 90. That means the dataset includes many ages, and no single age group dominates strongly. There is no clear distortion.

The blood pressure histogram also looks mostly uniform, roughly from 80 to 180. This indicates blood pressure values are spread evenly across this range, with no strong concentration around a specific value.

Overall, BMI looks more like real-world data because it has a bell-shaped distribution, while glucose, insulin, cholesterol, age, and blood pressure look more evenly distributed. When features are uniform, it can sometimes be harder for the model to find strong “natural patterns,” which may partly explain why our model performance stayed around ~0.50.

In [ ]:
import seaborn as sns

# ── Explicitly define numeric_cols ───────────────────────────────────────────
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

corr = X[numeric_cols].corr()

# ── Correlation Heatmap ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,          # show correlation values in each cell
    fmt=".2f",           # 2 decimal places
    cmap="coolwarm",     # diverging colormap for +/- correlations
    vmin=-1, vmax=1,     # full correlation range
    linewidths=0.5,      # cell separators
    ax=ax,
)
ax.set_title("Correlation Heatmap (Numeric Features)")
plt.tight_layout()
plt.show()

This correlation heatmap shows the linear relationship between each pair of numeric features in the dataset. The bright diagonal line appears because each feature is perfectly correlated with itself. Most cells outside the diagonal are dark, meaning the correlations are close to 0, which means that most numeric variables have weak or no strong linear relationship with each other. This shows low redundancy among features and that any patterns in the data may not be strongly linear.

In [ ]:
# Extract feature names from pipeline (after encoding)
prep = best_rf.named_steps["preprocessor"] # Corrected step name
model = best_rf.named_steps["classifier"] # Corrected step name

# Get original feature names for numerical features
numeric_feature_names = X.select_dtypes(include=np.number).columns.tolist()

# Get one-hot encoded feature names for categorical features
# We need to pass the original categorical feature names to get_feature_names_out
original_categorical_feature_names = X.select_dtypes(include="object").columns.tolist()
cat_transformer = prep.named_transformers_["cat"]
encoded_categorical_feature_names = cat_transformer.get_feature_names_out(original_categorical_feature_names).tolist()

# Combine all feature names in the order they appear in the transformed data
# ColumnTransformer orders them by the order of transformers, so numerical then categorical
all_feature_names = numeric_feature_names + encoded_categorical_feature_names

importances = model.feature_importances_

import pandas as pd
feat_imp = pd.DataFrame({"feature": all_feature_names, "importance": importances})
feat_imp = feat_imp.sort_values("importance", ascending=False)

top10 = feat_imp.head(10).sort_values("importance")

plt.figure(figsize=(8, 5))
plt.barh(top10["feature"], top10["importance"])
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("Top 10 Feature Importances:")
print(feat_imp.head(10).to_string(index=False))

This bar chart shows the 10 most important features used by the tuned Random Forest model when predicting diabetes. Features with longer bars have higher importance, meaning the model relied on them more often to make decisions across its trees. In our results, BMI and steps_per_day had the highest importance, followed by cholesterol, insulin, and glucose, suggesting these variables had the strongest influence on the model’s predictions compared to the other features.

In [ ]:
# ── DELIVERABLE 2: Confusion Matrix ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, rf_pred,
    display_labels=["No Diabetes", "Diabetes"],   # meaningful class names
    cmap="Blues",                                  # consistent colormap
    ax=ax,                                         # fixes orphaned plt.figure()
)
ax.set_title("Confusion Matrix — Tuned Random Forest")
plt.tight_layout()
plt.show()

The model correctly predicted 540 non-diabetic cases and 460 diabetic cases. However, it also misclassified 469 non-diabetic cases as diabetic and 531 diabetic cases as non-diabetic. Overall, the errors are still high, meaning the model has difficulty clearly separating the two classes.

In [ ]:
# ── DELIVERABLE 3: ROC Curve ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(
    y_test, rf_proba,
    name="Tuned Random Forest",
    ax=ax,                          # fixes the orphaned plt.figure() issue
)
ax.plot([0, 1], [0, 1], "k--", label="Random Classifier")   # baseline
ax.set_title("ROC Curve — Tuned Random Forest")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

This ROC curve shows the tradeoff between the True Positive Rate and the False Positive Rate at different probability thresholds for the tuned Random Forest. Ideally, the curve should be closer to the top-left corner, which indicates strong class separation. In our case, the curve is close to the diagonal line and the Area Under Curve is 0.50, meaning the model’s ability to distinguish between diabetic and non-diabetic cases is about the same as random guessing.

In [ ]:
# ── DELIVERABLE 4: Precision-Recall Curve ───────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
PrecisionRecallDisplay.from_predictions(
    y_test, rf_proba,
    name="Tuned Random Forest",
    ax=ax,                        # fixes the orphaned plt.figure() issue
)
ax.set_title("Precision-Recall Curve — Tuned Random Forest")
plt.tight_layout()
plt.show()

The curve shows the relationship between Precision of how many predicted diabetics are truly diabetic and Recall of how many actual diabetics the model correctly finds for different probability thresholds. A better model would keep precision high even as recall increases. In our graph, precision stays around 0.50 for most recall values, and the Average Precision (AP) is about 0.51, which is only slightly above chance. This indicates the model does not strongly separate diabetic cases from non-diabetic cases when using probability thresholds.

In [ ]:
residual = y_test.values - rf_proba

# ── DELIVERABLE 6: Residual-like Scatter Plot ────────────────────────────────
plt.figure(figsize=(7, 5))
plt.scatter(rf_proba, residual, s=10, alpha=0.5, color="steelblue")
plt.axhline(0, color="red", linestyle="--", linewidth=1, label="Zero residual")
plt.title("Residual-like Plot: (y − Predicted Probability) vs Predicted Probability")
plt.xlabel("Predicted Probability (diabetes = 1)")
plt.ylabel("Residual-like Error (y − prob)")
plt.legend()
plt.tight_layout()
plt.show()

# ── DELIVERABLE 7: Histogram of Residual-like Errors ────────────────────────
plt.figure(figsize=(7, 5))
plt.hist(residual, bins=30, color="steelblue", edgecolor="white")
plt.axvline(0, color="red", linestyle="--", linewidth=1, label="Zero residual")
plt.title("Histogram of Residual-like Errors (y − Predicted Probability)")
plt.xlabel("Residual-like Error (y − prob)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

The scatter plot shows two main bands: one positive band when the true class is 1 and one negative band when the true class is 0, because residuals for class 1 are 1 − 𝑝 1−p and for class 0 are 0 − 𝑝 0−p. In our plot, most predicted probabilities are clustered around roughly 0.35 to 0.65, and the errors are not centered tightly around 0, which suggests the model is often uncertain and does not assign very confident probabilities. The histogram shows two separated groups of residuals, reflecting the two classes, and indicates that many predictions still produce sizable errors, meaning the probability estimates are not very accurate or well-separated.

In [ ]:
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

print("=" * 55)
print("       Tuned Random Forest — Classification Report")
print("=" * 55)
print(classification_report(y_test, rf_pred, target_names=["No Diabetes", "Diabetes"]))

# ── DELIVERABLE 2: Confusion Matrix ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, rf_pred,
    display_labels=["No Diabetes", "Diabetes"],
    cmap="Blues",
    ax=ax,
)
ax.set_title("Confusion Matrix — Tuned Random Forest")
plt.tight_layout()
plt.show()

# ── DELIVERABLE 3: ROC Curve ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(
    y_test, rf_proba,
    name="Tuned Random Forest",
    ax=ax,
)
ax.plot([0, 1], [0, 1], "k--", label="Random Classifier")
ax.set_title("ROC Curve — Tuned Random Forest")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# ── DELIVERABLE 4: Precision-Recall Curve ───────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
PrecisionRecallDisplay.from_predictions(
    y_test, rf_proba,
    name="Tuned Random Forest",
    ax=ax,
)
ax.set_title("Precision-Recall Curve — Tuned Random Forest")
plt.tight_layout()
plt.show()

# ── DELIVERABLE 5: Feature Importances ──────────────────────────────────────
# Works if the pipeline's final step is named 'classifier' or is the last step

# Extract preprocessing step from the pipeline
prep = best_rf.named_steps["preprocessor"]
model = best_rf.named_steps["classifier"]

# Get original feature names for numerical features
numeric_feature_names = X.select_dtypes(include=np.number).columns.tolist()

# Get one-hot encoded feature names for categorical features
original_categorical_feature_names = X.select_dtypes(include="object").columns.tolist()
cat_transformer = prep.named_transformers_["cat"]
encoded_categorical_feature_names = cat_transformer.get_feature_names_out(original_categorical_feature_names).tolist()

# Combine all feature names in the order they appear in the transformed data
# ColumnTransformer orders them by the order of transformers, so numerical then categorical
all_feature_names = numeric_feature_names + encoded_categorical_feature_names

rf_step = best_rf.steps[-1][1]          # last step of the pipeline

importances = pd.Series(rf_step.feature_importances_, index=all_feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
importances.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Feature Importances — Tuned Random Forest")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
plt.show()